In [7]:
import pandas as pd
import sqlite3
import json

In [9]:
movies_df = pd.read_csv("../data/raw/tmdb_5000_movies.csv")
movies_df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [11]:
credits_df = pd.read_csv("../data/raw/tmdb_5000_credits.csv")
credits_df.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [13]:
# Create connection to SQLite database
conn = sqlite3.connect("../data/cineanalytica.db")
cursor = conn.cursor()

print("Database created successfully!")

Database created successfully!


In [15]:
cursor.executescript("""
DROP TABLE IF EXISTS movies;

CREATE TABLE movies (
    movie_id INTEGER PRIMARY KEY,
    title TEXT,
    original_title TEXT,
    original_language TEXT,
    release_date TEXT,
    runtime REAL,
    budget REAL,
    revenue REAL,
    popularity REAL,
    vote_average REAL,
    vote_count REAL,
    overview TEXT
);
""")

conn.commit()
print("✅ movies table created")

✅ movies table created


In [17]:
# Insert data into movies table

for _, row in movies_df.iterrows():
    cursor.execute("""
        INSERT INTO movies (
            movie_id, title, original_title, original_language,
            release_date, runtime, budget, revenue,
            popularity, vote_average, vote_count, overview
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        row["id"],
        row["title"],
        row["original_title"],
        row["original_language"],
        row["release_date"],
        row["runtime"],
        row["budget"],
        row["revenue"],
        row["popularity"],
        row["vote_average"],
        row["vote_count"],
        row["overview"]
    ))

conn.commit()

print("✅ Movie data inserted successfully!")

✅ Movie data inserted successfully!


In [19]:
cursor.execute("SELECT COUNT(*) FROM movies;")
cursor.fetchone()

(4803,)

In [21]:
cursor.executescript("""
DROP TABLE IF EXISTS actors;

CREATE TABLE actors (
    actor_id INTEGER PRIMARY KEY,
    actor_name TEXT
);
""")

conn.commit()
print("✅ actors table created")

✅ actors table created


In [23]:
cursor.executescript("""
DROP TABLE IF EXISTS movie_cast;

CREATE TABLE movie_cast (
    movie_id INTEGER,
    actor_id INTEGER,
    character_name TEXT,
    cast_order INTEGER,
    PRIMARY KEY (movie_id, actor_id),
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id),
    FOREIGN KEY (actor_id) REFERENCES actors(actor_id)
);
""")

conn.commit()
print("✅ movie_cast table created")

✅ movie_cast table created


In [25]:
# Insert cast data into actors and movie_cast

actor_seen = set()  # to avoid inserting same actor twice

for _, row in credits_df.iterrows():
    movie_id = int(row["movie_id"])
    cast_list = json.loads(row["cast"])  # convert string -> list of dicts
    
    for c in cast_list:
        actor_id = int(c["id"])
        actor_name = c["name"]
        character_name = c.get("character", "")
        cast_order = c.get("order", None)

        # Insert into actors table only once
        if actor_id not in actor_seen:
            cursor.execute(
                "INSERT OR IGNORE INTO actors (actor_id, actor_name) VALUES (?, ?)",
                (actor_id, actor_name)
            )
            actor_seen.add(actor_id)

        # Insert into movie_cast table
        cursor.execute(
            "INSERT OR IGNORE INTO movie_cast (movie_id, actor_id, character_name, cast_order) VALUES (?, ?, ?, ?)",
            (movie_id, actor_id, character_name, cast_order)
        )

conn.commit()
print("✅ Cast data inserted into actors + movie_cast!")

✅ Cast data inserted into actors + movie_cast!


In [27]:
cursor.execute("SELECT COUNT(*) FROM actors;")
actors_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM movie_cast;")
cast_count = cursor.fetchone()[0]

actors_count, cast_count

(54588, 106084)

In [29]:
cursor.executescript("""
DROP TABLE IF EXISTS directors;

CREATE TABLE directors (
    director_id INTEGER PRIMARY KEY,
    director_name TEXT
);
""")

conn.commit()
print("✅ directors table created")

✅ directors table created


In [31]:
cursor.executescript("""
DROP TABLE IF EXISTS movie_directors;

CREATE TABLE movie_directors (
    movie_id INTEGER,
    director_id INTEGER,
    PRIMARY KEY (movie_id, director_id),
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id),
    FOREIGN KEY (director_id) REFERENCES directors(director_id)
);
""")

conn.commit()
print("✅ movie_directors table created")

✅ movie_directors table created


In [33]:
director_seen = set()

for _, row in credits_df.iterrows():
    movie_id = int(row["movie_id"])
    crew_list = json.loads(row["crew"])  # string -> list of dicts
    
    for c in crew_list:
        if c.get("job") == "Director":
            director_id = int(c["id"])
            director_name = c["name"]

            # Insert director only once
            if director_id not in director_seen:
                cursor.execute(
                    "INSERT OR IGNORE INTO directors (director_id, director_name) VALUES (?, ?)",
                    (director_id, director_name)
                )
                director_seen.add(director_id)

            # Link movie to director
            cursor.execute(
                "INSERT OR IGNORE INTO movie_directors (movie_id, director_id) VALUES (?, ?)",
                (movie_id, director_id)
            )

conn.commit()
print("✅ Directors inserted!")

✅ Directors inserted!


In [35]:
cursor.execute("SELECT COUNT(*) FROM directors;")
directors_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM movie_directors;")
movie_directors_count = cursor.fetchone()[0]

directors_count, movie_directors_count

(2578, 5166)

In [37]:
query = """
SELECT m.title, d.director_name
FROM movies m
JOIN movie_directors md ON m.movie_id = md.movie_id
JOIN directors d ON md.director_id = d.director_id
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,title,director_name
0,Avatar,James Cameron
1,Pirates of the Caribbean: At World's End,Gore Verbinski
2,Spectre,Sam Mendes
3,The Dark Knight Rises,Christopher Nolan
4,John Carter,Andrew Stanton
5,Spider-Man 3,Sam Raimi
6,Tangled,Byron Howard
7,Tangled,Nathan Greno
8,Avengers: Age of Ultron,Joss Whedon
9,Harry Potter and the Half-Blood Prince,David Yates


In [39]:
import sys
print(sys.version)

3.12.12 (main, Oct  9 2025, 11:07:00) [Clang 17.0.0 (clang-1700.6.3.2)]


In [41]:
import surprise
import shap
print("All good 🚀")

All good 🚀


In [43]:
cursor.executescript("""
DROP TABLE IF EXISTS movie_genres;
DROP TABLE IF EXISTS genres;

CREATE TABLE genres (
  genre_id INTEGER PRIMARY KEY,
  genre_name TEXT NOT NULL UNIQUE
);

CREATE TABLE movie_genres (
  movie_id INTEGER NOT NULL,
  genre_id INTEGER NOT NULL,
  PRIMARY KEY (movie_id, genre_id),
  FOREIGN KEY (movie_id) REFERENCES movies(movie_id),
  FOREIGN KEY (genre_id) REFERENCES genres(genre_id)
);
""")
conn.commit()
print("✅ genres + movie_genres tables created")

✅ genres + movie_genres tables created


In [45]:
import json

genre_seen = set()

for _, row in movies_df.iterrows():
    movie_id = int(row["id"])
    if pd.isna(row["genres"]):
        continue

    genres_list = json.loads(row["genres"])  # list of dicts: [{"id": 28, "name": "Action"}, ...]

    for g in genres_list:
        genre_id = int(g["id"])
        genre_name = g["name"].strip()

        # insert genre once
        if genre_id not in genre_seen:
            cursor.execute(
                "INSERT OR IGNORE INTO genres (genre_id, genre_name) VALUES (?, ?)",
                (genre_id, genre_name)
            )
            genre_seen.add(genre_id)

        # link movie <-> genre
        cursor.execute(
            "INSERT OR IGNORE INTO movie_genres (movie_id, genre_id) VALUES (?, ?)",
            (movie_id, genre_id)
        )

conn.commit()
print("✅ Genre data inserted into genres + movie_genres")

✅ Genre data inserted into genres + movie_genres


In [47]:
cursor.execute("SELECT COUNT(*) FROM genres;")
genres_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM movie_genres;")
movie_genres_count = cursor.fetchone()[0]

genres_count, movie_genres_count

(20, 12160)

In [49]:
query = """
SELECT m.title, group_concat(g.genre_name, ', ') AS genres
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g ON mg.genre_id = g.genre_id
GROUP BY m.movie_id
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,title,genres
0,Four Rooms,"Crime, Comedy"
1,Star Wars,"Adventure, Action, Science Fiction"
2,Finding Nemo,"Animation, Family"
3,Forrest Gump,"Comedy, Drama, Romance"
4,American Beauty,Drama
5,Dancer in the Dark,"Drama, Crime, Music"
6,The Fifth Element,"Adventure, Fantasy, Action, Thriller, Science ..."
7,Metropolis,"Drama, Science Fiction"
8,My Life Without Me,"Drama, Romance"
9,Pirates of the Caribbean: The Curse of the Bla...,"Adventure, Fantasy, Action"


In [51]:
ratings_df = pd.read_csv("../data/raw/ratings.csv")
ratings_df.head(), ratings_df.shape

(   userId  movieId  rating  timestamp
 0       1        1     4.0  964982703
 1       1        3     4.0  964981247
 2       1        6     4.0  964982224
 3       1       47     5.0  964983815
 4       1       50     5.0  964982931,
 (100836, 4))

In [53]:
ratings_df.columns

Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='object')

In [55]:
import os
sorted(os.listdir("../data/raw"))

['IMDB Dataset.csv',
 'movies.csv',
 'ratings.csv',
 'tmdb_5000_credits.csv',
 'tmdb_5000_movies.csv']

In [59]:
import os
sorted(os.listdir("../data/raw"))

['IMDB Dataset.csv',
 'links.csv',
 'movies.csv',
 'ratings.csv',
 'tmdb_5000_credits.csv',
 'tmdb_5000_movies.csv']

In [61]:
links_df = pd.read_csv("../data/raw/links.csv")
links_df.head(), links_df.shape

(   movieId  imdbId   tmdbId
 0        1  114709    862.0
 1        2  113497   8844.0
 2        3  113228  15602.0
 3        4  114885  31357.0
 4        5  113041  11862.0,
 (9742, 3))

In [63]:
links_df = links_df.dropna(subset=["tmdbId"])
links_df["tmdbId"] = links_df["tmdbId"].astype(int)

links_df.head(), links_df.shape

(   movieId  imdbId  tmdbId
 0        1  114709     862
 1        2  113497    8844
 2        3  113228   15602
 3        4  114885   31357
 4        5  113041   11862,
 (9734, 3))

In [65]:
ratings_mapped = ratings_df.merge(
    links_df[["movieId", "tmdbId"]],
    on="movieId",
    how="inner"
)

ratings_mapped.rename(columns={
    "userId": "user_id",
    "tmdbId": "movie_id"
}, inplace=True)

ratings_mapped = ratings_mapped[["user_id", "movie_id", "rating", "timestamp"]]

ratings_mapped.head(), ratings_mapped.shape

(   user_id  movie_id  rating  timestamp
 0        1       862     4.0  964982703
 1        1     15602     4.0  964981247
 2        1       949     4.0  964982224
 3        1       807     5.0  964983815
 4        1       629     5.0  964982931,
 (100823, 4))

In [67]:
cursor.executescript("""
DROP TABLE IF EXISTS ratings;

CREATE TABLE ratings (
    user_id INTEGER,
    movie_id INTEGER,
    rating REAL,
    timestamp INTEGER,
    PRIMARY KEY (user_id, movie_id),
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
);
""")

conn.commit()
print("✅ ratings table created")

✅ ratings table created


In [69]:
cursor.executemany(
    "INSERT OR REPLACE INTO ratings (user_id, movie_id, rating, timestamp) VALUES (?, ?, ?, ?)",
    list(ratings_mapped.itertuples(index=False, name=None))
)

conn.commit()
print("✅ ratings inserted:", len(ratings_mapped))

✅ ratings inserted: 100823


In [71]:
pd.read_sql_query("""
SELECT m.title, COUNT(*) AS num_ratings, AVG(r.rating) AS avg_rating
FROM ratings r
JOIN movies m ON m.movie_id = r.movie_id
GROUP BY r.movie_id
ORDER BY num_ratings DESC
LIMIT 10;
""", conn)

,title,num_ratings,avg_rating
0,Forrest Gump,329,4.164134
1,The Shawshank Redemption,317,4.429022
2,Pulp Fiction,307,4.197068
3,The Silence of the Lambs,279,4.161290
4,The Matrix,278,4.192446
5,Star Wars,251,4.231076
6,Jurassic Park,238,3.750000
7,Braveheart,237,4.031646
8,Terminator 2: Judgment Day,224,3.970982
9,Schindler's List,220,4.225000


In [73]:
cursor.executescript("""
CREATE INDEX IF NOT EXISTS idx_ratings_movie_id ON ratings(movie_id);
CREATE INDEX IF NOT EXISTS idx_ratings_user_id ON ratings(user_id);
""")
conn.commit()
print("✅ Indexes created")

✅ Indexes created


In [77]:
###1) How many unique users & movies have ratings?

In [79]:
pd.read_sql_query("""
SELECT 
  COUNT(DISTINCT user_id) AS unique_users,
  COUNT(DISTINCT movie_id) AS unique_movies,
  COUNT(*) AS total_ratings
FROM ratings;
""", conn)

,unique_users,unique_movies,total_ratings
0,610,9715,100822


In [81]:
### 2) Confirm no orphan movie_ids (ratings that don’t match movies)

In [83]:
pd.read_sql_query("""
SELECT COUNT(*) AS orphan_ratings
FROM ratings r
LEFT JOIN movies m ON m.movie_id = r.movie_id
WHERE m.movie_id IS NULL;
""", conn)

,orphan_ratings
0,30629


In [85]:
valid_movie_ids = pd.read_sql_query(
    "SELECT movie_id FROM movies",
    conn
)["movie_id"].tolist()

ratings_filtered = ratings_mapped[
    ratings_mapped["movie_id"].isin(valid_movie_ids)
]

ratings_filtered.shape

(70194, 4)

In [87]:
cursor.execute("DELETE FROM ratings;")
conn.commit()

cursor.executemany(
    "INSERT OR REPLACE INTO ratings (user_id, movie_id, rating, timestamp) VALUES (?, ?, ?, ?)",
    list(ratings_filtered.itertuples(index=False, name=None))
)

conn.commit()
print("✅ Clean ratings inserted:", len(ratings_filtered))

✅ Clean ratings inserted: 70194


In [89]:
pd.read_sql_query("""
SELECT COUNT(*) AS orphan_ratings
FROM ratings r
LEFT JOIN movies m ON m.movie_id = r.movie_id
WHERE m.movie_id IS NULL;
""", conn)

,orphan_ratings
0,0


In [91]:
ratings_model_df = pd.read_sql_query("""
SELECT
  r.user_id,
  r.movie_id,
  r.rating,
  m.title
FROM ratings r
JOIN movies m ON m.movie_id = r.movie_id
""", conn)

ratings_model_df.head(), ratings_model_df.shape

(   user_id  movie_id  rating                title
 0        1       862     4.0            Toy Story
 1        1       807     5.0                Se7en
 2        1       629     5.0   The Usual Suspects
 3        1       755     3.0  From Dusk Till Dawn
 4        1     13685     5.0        Bottle Rocket,
 (70193, 4))

In [93]:
ratings_model_df["user_id"].nunique(), ratings_model_df["movie_id"].nunique(), len(ratings_model_df)

(610, 3535, 70193)

In [97]:
### (Optional but very useful) build the user–item matrix for Surprise later:

In [95]:
user_item = ratings_model_df.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

user_item.shape

(610, 3535)

In [99]:
import os
sorted(os.listdir("../data/raw"))

['IMDB Dataset.csv',
 'links.csv',
 'movies.csv',
 'ratings.csv',
 'tmdb_5000_credits.csv',
 'tmdb_5000_movies.csv']

In [101]:
cursor.executescript("""
DROP TABLE IF EXISTS reviews;

CREATE TABLE reviews (
    review_id INTEGER PRIMARY KEY AUTOINCREMENT,
    review_text TEXT NOT NULL,
    sentiment TEXT NOT NULL
);
""")
conn.commit()
print("✅ reviews table created")

✅ reviews table created


In [103]:
import pandas as pd

imdb_path = "../data/raw/IMDB Dataset.csv"
imdb_df = pd.read_csv(imdb_path)

imdb_df.columns = [c.strip().lower() for c in imdb_df.columns]
imdb_df = imdb_df.rename(columns={"review": "review_text"})  # standardize

imdb_df = imdb_df[["review_text", "sentiment"]].dropna()
imdb_df["sentiment"] = imdb_df["sentiment"].astype(str).str.lower().str.strip()

imdb_df.head(), imdb_df.shape

(                                         review_text sentiment
 0  One of the other reviewers has mentioned that ...  positive
 1  A wonderful little production. <br /><br />The...  positive
 2  I thought this was a wonderful way to spend ti...  positive
 3  Basically there's a family where a little boy ...  negative
 4  Petter Mattei's "Love in the Time of Money" is...  positive,
 (50000, 2))

In [105]:
records = list(imdb_df.itertuples(index=False, name=None))

cursor.executemany(
    "INSERT INTO reviews (review_text, sentiment) VALUES (?, ?);",
    records
)
conn.commit()

print(f"✅ Inserted {len(records)} reviews")

✅ Inserted 50000 reviews


In [107]:
pd.read_sql_query("""
SELECT sentiment, COUNT(*) AS cnt
FROM reviews
GROUP BY sentiment
ORDER BY cnt DESC;
""", conn)

,sentiment,cnt
0,positive,25000
1,negative,25000


In [109]:
import pandas as pd
tmdb = pd.read_csv("../data/raw/tmdb_5000_movies.csv")
tmdb.columns = [c.strip().lower() for c in tmdb.columns]
tmdb["release_date"] = pd.to_datetime(tmdb.get("release_date"), errors="coerce")
tmdb["release_year"] = tmdb["release_date"].dt.year
tmdb["release_month"] = tmdb["release_date"].dt.month

base_features = tmdb[["id","title","budget","revenue","runtime","popularity","vote_average","vote_count","release_year","release_month"]].rename(columns={"id":"tmdb_id"})
base_features.head()

,tmdb_id,title,budget,revenue,runtime,popularity,vote_average,vote_count,release_year,release_month
0,19995,Avatar,237000000,2787965087,162.0,150.437577,7.2,11800,2009.0,12.0
1,285,Pirates of the Caribbean: At World's End,300000000,961000000,169.0,139.082615,6.9,4500,2007.0,5.0
2,206647,Spectre,245000000,880674609,148.0,107.376788,6.3,4466,2015.0,10.0
3,49026,The Dark Knight Rises,250000000,1084939099,165.0,112.312950,7.6,9106,2012.0,7.0
4,49529,John Carter,260000000,284139100,132.0,43.926995,6.1,2124,2012.0,3.0


In [111]:
ratings_agg = pd.read_sql_query("""
SELECT movie_id, COUNT(*) AS num_ratings, AVG(rating) AS avg_rating
FROM ratings
GROUP BY movie_id
""", conn)
ratings_agg.head()

,movie_id,num_ratings,avg_rating
0,5,20,3.700000
1,11,251,4.231076
2,12,141,3.960993
3,13,329,4.164134
4,14,204,4.056373


In [113]:
genres_df = pd.read_sql_query("""
SELECT mg.movie_id, g.genre_name
FROM movie_genres mg
JOIN genres g ON g.genre_id = mg.genre_id
""", conn)

genre_ohe = pd.crosstab(genres_df["movie_id"], genres_df["genre_name"]).reset_index().rename(columns={"movie_id":"movie_id"})
genre_ohe.head()

genre_name,movie_id,Action,Adventure,Animation,Comedy,Crime,Documentary,Drama,Family,Fantasy,...,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
0,5,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,11,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,12,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,13,0,0,0,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0
4,14,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [115]:
pd.read_sql_query("SELECT movie_id, title FROM movies LIMIT 5;", conn)

,movie_id,title
0,5,Four Rooms
1,11,Star Wars
2,12,Finding Nemo
3,13,Forrest Gump
4,14,American Beauty


In [117]:
# 1️⃣ Base TMDB features
tmdb = pd.read_csv("../data/raw/tmdb_5000_movies.csv")
tmdb.columns = [c.strip().lower() for c in tmdb.columns]

tmdb["release_date"] = pd.to_datetime(tmdb["release_date"], errors="coerce")
tmdb["release_year"] = tmdb["release_date"].dt.year
tmdb["release_month"] = tmdb["release_date"].dt.month

base_features = tmdb[[
    "id", "title", "budget", "revenue", "runtime",
    "popularity", "vote_average", "vote_count",
    "release_year", "release_month"
]].rename(columns={"id": "movie_id"})

# 2️⃣ Ratings aggregates from SQLite
ratings_agg = pd.read_sql_query("""
SELECT movie_id,
       COUNT(*) AS num_ratings,
       AVG(rating) AS avg_rating
FROM ratings
GROUP BY movie_id;
""", conn)

# 3️⃣ Genre one-hot encoding from SQLite
genres_df = pd.read_sql_query("""
SELECT mg.movie_id, g.genre_name
FROM movie_genres mg
JOIN genres g ON g.genre_id = mg.genre_id;
""", conn)

genre_ohe = pd.crosstab(
    genres_df["movie_id"],
    genres_df["genre_name"]
).reset_index()

# 4️⃣ Merge everything
movie_features = base_features.merge(
    ratings_agg,
    on="movie_id",
    how="left"
)

movie_features = movie_features.merge(
    genre_ohe,
    on="movie_id",
    how="left"
)

# Fill missing values
movie_features["num_ratings"] = movie_features["num_ratings"].fillna(0).astype(int)
movie_features["avg_rating"] = movie_features["avg_rating"].fillna(0)

genre_cols = [c for c in genre_ohe.columns if c != "movie_id"]
movie_features[genre_cols] = movie_features[genre_cols].fillna(0).astype(int)

movie_features.head(), movie_features.shape

(   movie_id                                     title     budget     revenue  \
 0     19995                                    Avatar  237000000  2787965087   
 1       285  Pirates of the Caribbean: At World's End  300000000   961000000   
 2    206647                                   Spectre  245000000   880674609   
 3     49026                     The Dark Knight Rises  250000000  1084939099   
 4     49529                               John Carter  260000000   284139100   
 
    runtime  popularity  vote_average  vote_count  release_year  release_month  \
 0    162.0  150.437577           7.2       11800        2009.0           12.0   
 1    169.0  139.082615           6.9        4500        2007.0            5.0   
 2    148.0  107.376788           6.3        4466        2015.0           10.0   
 3    165.0  112.312950           7.6        9106        2012.0            7.0   
 4    132.0   43.926995           6.1        2124        2012.0            3.0   
 
    ...  History  

In [119]:
movie_features.to_sql("movie_features", conn, if_exists="replace", index=False)

pd.read_sql_query("""
SELECT COUNT(*) AS total_movies
FROM movie_features;
""", conn)

,total_movies
0,4803


In [121]:
pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name;
""", conn)

,name
0,actors
1,directors
2,genres
3,movie_cast
4,movie_directors
5,movie_features
6,movie_genres
7,movies
8,ratings
9,reviews


In [123]:
pd.read_sql_query("PRAGMA table_info(movies);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,movie_id,INTEGER,0,None,1
1,1,title,TEXT,0,None,0
2,2,original_title,TEXT,0,None,0
3,3,original_language,TEXT,0,None,0
4,4,release_date,TEXT,0,None,0
5,5,runtime,REAL,0,None,0
6,6,budget,REAL,0,None,0
7,7,revenue,REAL,0,None,0
8,8,popularity,REAL,0,None,0
9,9,vote_average,REAL,0,None,0


In [125]:
pd.read_sql_query("PRAGMA table_info(movie_cast);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,movie_id,INTEGER,0,None,1
1,1,actor_id,INTEGER,0,None,2
2,2,character_name,TEXT,0,None,0
3,3,cast_order,INTEGER,0,None,0


In [127]:
pd.read_sql_query("""
SELECT mc.movie_id, m.title, mc.actor_id, a.actor_name, mc.cast_order
FROM movie_cast mc
JOIN movies m ON m.movie_id = mc.movie_id
JOIN actors a ON a.actor_id = mc.actor_id
WHERE mc.movie_id IN (5, 11, 12, 13, 14)
ORDER BY mc.movie_id, mc.cast_order
LIMIT 30;
""", conn)

,movie_id,title,actor_id,actor_name,cast_order
0,5,Four Rooms,3129,Tim Roth,0
1,5,Four Rooms,3131,Antonio Banderas,1
2,5,Four Rooms,3130,Jennifer Beals,2
3,5,Four Rooms,3125,Madonna,3
4,5,Four Rooms,3141,Marisa Tomei,4
5,5,Four Rooms,62,Bruce Willis,5
6,5,Four Rooms,138,Quentin Tarantino,6
7,5,Four Rooms,3122,Sammi Davis,7
8,5,Four Rooms,3123,Amanda de Cadenet,8
9,5,Four Rooms,3124,Valeria Golino,9


In [129]:
import pandas as pd

# 1️⃣ Get top-billed actors only (cast_order = 0)
top_cast = pd.read_sql_query("""
SELECT
    mc.movie_id,
    mc.actor_id,
    m.release_date,
    m.revenue
FROM movie_cast mc
JOIN movies m ON m.movie_id = mc.movie_id
WHERE mc.cast_order = 0
AND m.revenue IS NOT NULL
AND m.release_date IS NOT NULL;
""", conn)

# Convert release_date to datetime
top_cast["release_date"] = pd.to_datetime(top_cast["release_date"], errors="coerce")

# Sort properly for leakage-safe computation
top_cast = top_cast.sort_values(["actor_id", "release_date"])

top_cast.head()

,movie_id,actor_id,release_date,revenue
2872,11,2,1977-05-25,775398007.0
1971,1891,2,1980-05-17,538400000.0
1478,1892,2,1983-05-23,572700000.0
3556,17339,3,1978-11-01,7230000.0
2065,85,3,1981-06-12,389925971.0


In [131]:
# 2️⃣ Leakage-safe star power: avg revenue of PREVIOUS movies only
top_cast["star_power"] = (
    top_cast.groupby("actor_id")["revenue"]
            .apply(lambda s: s.shift(1).expanding().mean())
            .reset_index(level=0, drop=True)
)

top_cast[["movie_id", "actor_id", "release_date", "revenue", "star_power"]].head(10)

,movie_id,actor_id,release_date,revenue,star_power
2872,11,2,1977-05-25,775398007.0,NaN
1971,1891,2,1980-05-17,538400000.0,7.753980e+08
1478,1892,2,1983-05-23,572700000.0,6.568990e+08
3556,17339,3,1978-11-01,7230000.0,NaN
2065,85,3,1981-06-12,389925971.0,7.230000e+06
1708,78,3,1982-06-25,33139618.0,1.985780e+08
1680,87,3,1984-05-23,333000000.0,1.434319e+08
2781,9281,3,1985-02-08,68706993.0,1.908239e+08
998,89,3,1989-05-24,474171806.0,1.664005e+08
1041,9869,3,1992-06-04,178051587.0,2.176957e+08


In [133]:
# 3️⃣ Replace NaN (first movie per actor) with 0
top_cast["star_power"] = top_cast["star_power"].fillna(0)

# 4️⃣ Keep only movie_id + star_power
movie_star_power = top_cast[["movie_id", "star_power"]]

# Since we only took cast_order = 0, there is one actor per movie.
# But let's ensure no duplicates:
movie_star_power = movie_star_power.groupby("movie_id", as_index=False)["star_power"].mean()

movie_star_power.head()

,movie_id,star_power
0,5,0.000000e+00
1,11,0.000000e+00
2,12,0.000000e+00
3,13,1.339068e+08
4,14,2.447379e+08


In [135]:
# Merge into movie_features dataframe
movie_features = movie_features.merge(
    movie_star_power,
    on="movie_id",
    how="left"
)

# Replace any remaining NaN with 0
movie_features["star_power"] = movie_features["star_power"].fillna(0)

movie_features[["movie_id", "star_power"]].head()

,movie_id,star_power
0,19995,0.000000e+00
1,285,1.718870e+08
2,206647,3.683570e+08
3,49026,1.966893e+08
4,49529,0.000000e+00


In [137]:
movie_features.to_sql("movie_features", conn, if_exists="replace", index=False)

pd.read_sql_query("""
SELECT movie_id, star_power
FROM movie_features
ORDER BY star_power DESC
LIMIT 5;
""", conn)

,movie_id,star_power
0,18823,2.787965e+09
1,49527,1.510339e+09
2,86825,1.025491e+09
3,57165,1.022300e+09
4,1440,9.900171e+08


In [139]:
import numpy as np

movie_features["log_star_power"] = np.log1p(movie_features["star_power"])

# Drop raw star_power (cleaner for modeling)
movie_features = movie_features.drop(columns=["star_power"])

movie_features[["movie_id", "log_star_power"]].head()

,movie_id,log_star_power
0,19995,0.000000
1,285,18.962348
2,206647,19.724563
3,49026,19.097136
4,49529,0.000000


In [141]:
movie_features.to_sql("movie_features", conn, if_exists="replace", index=False)

pd.read_sql_query("""
SELECT movie_id, log_star_power
FROM movie_features
ORDER BY log_star_power DESC
LIMIT 5;
""", conn)

,movie_id,log_star_power
0,18823,21.748578
1,49527,21.135600
2,86825,20.748437
3,57165,20.745321
4,1440,20.713233
